# Layout algorithm lab

This notebook loads complete saved layouts from `data/training/`, uses each file's `wysokosc_rolki` as the authoritative generated result, compares other algorithms on the same inputs, and shows result tables, SVG previews, and comparison plots.

In [ ]:
from pathlib import Path
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from IPython.display import HTML, SVG, display

ROOT = Path.cwd()
if not (ROOT / 'optimizer').exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))
from experiments.notebooks.layout_lab_utils import (
    area_decreasing, benchmark, first_fit_decreasing, max_rects,
    genetic_baseline,
    load_json, load_training_layout, random_search,
)

In [ ]:
layout_paths = sorted((ROOT / 'data/training').glob('*.json'))
if not layout_paths:
    raise ValueError('No saved layouts found in data/training/.')
saved_cases = []
for layout_path in layout_paths:
    layout = load_json(layout_path)
    machine_width, items = load_training_layout(layout_path)
    saved_cases.append({
        'case_id': layout_path.stem,
        'layout_path': layout_path,
        'layout': layout,
        'machine_width': machine_width,
        'items': items,
    })
print(f'Saved layouts loaded: {len(saved_cases)}')

In [ ]:
# Create a fresh persistent table containing the saved genetic benchmark only.
RESULTS_TABLE_PATH = ROOT / "experiments/layout_algorithm_results.csv"

benchmark_records = {
    record["case_id"]: record
    for record in load_json(ROOT / "experiments/benchmark_results.json")
}

benchmark_rows = []
for case in saved_cases:
    layout = case["layout"]
    rectangles = layout.get("elementy", [])
    saved_height = float(layout["wysokosc_rolki"])
    used_area = sum(
        float(item.get("w", 0)) * float(item.get("h", 0))
        for item in rectangles
    )
    benchmark_record = benchmark_records.get(case["case_id"], {})
    benchmark_rows.append({
        "case_id": case["case_id"],
        "algorithm": "Saved genetic layout",
        "height_mm": saved_height,
        "used_area_mm2": used_area,
        "utilization_percent": (
            used_area / (case["machine_width"] * saved_height) * 100
            if saved_height else 0.0
        ),
        "placed": len(rectangles),
        "missing": 0,
        "seconds": benchmark_record.get("baseline_seconds", np.nan),
        "machine_width": case["machine_width"],
        "layout_path": str(case["layout_path"]),
    })

results_df = pd.DataFrame(benchmark_rows)
height_categories = pd.qcut(
    results_df.set_index("case_id")["height_mm"].rank(method="first"),
    q=3,
    labels=["Short", "Medium", "Tall"],
)
results_df["genetic_height_category"] = results_df["case_id"].map(height_categories)
results_df.to_csv(RESULTS_TABLE_PATH, index=False)


In [ ]:
def add_algorithm_results(algorithm, algorithm_name, **benchmark_options):
    """Run one algorithm, replace its previous rows, and save the newest table."""
    global results_df

    rows = []
    for case in saved_cases:
        result = benchmark(
            case["items"],
            case["machine_width"],
            algorithms={algorithm_name: algorithm},
            **benchmark_options,
        )[0]
        rows.append({
            "case_id": case["case_id"],
            "algorithm": algorithm_name,
            "height_mm": result["height_mm"],
            "used_area_mm2": result["used_area_mm2"],
            "utilization_percent": result["utilization_percent"],
            "placed": result["placed"],
            "missing": result["missing"],
            "seconds": result["seconds"],
            "machine_width": case["machine_width"],
            "layout_path": str(case["layout_path"]),
            "genetic_height_category": results_df.loc[
                results_df["case_id"] == case["case_id"],
                "genetic_height_category",
            ].iloc[0],
        })

    new_rows = pd.DataFrame(rows)
    results_df = pd.concat(
        [
            results_df[results_df["algorithm"] != algorithm_name],
            new_rows,
        ],
        ignore_index=True,
    )
    results_df.to_csv(RESULTS_TABLE_PATH, index=False)
    return results_df


In [ ]:
def run_algorithm_sequential(algorithm, algorithm_name, legacy_algorithm=genetic_baseline, **benchmark_options):
    """Run the new algorithm and legacy genetic baseline case by case."""
    ordered_cases = sorted(
        saved_cases,
        key=lambda case: (case["layout_path"].name.startswith("gen_"), case["layout_path"].name.lower()),
    )
    new_results = []
    new_total_started = time.perf_counter()
    legacy_total_elapsed = 0.0
    for index, case in enumerate(ordered_cases, start=1):
        print(f"[{index}/{len(ordered_cases)}] {case['layout_path'].name}")
        population_size = benchmark_options.get("population_size", 40)
        generations = benchmark_options.get("generations", 80)
        configured_algorithm = lambda items, width: algorithm(
            items,
            width,
            population_size=population_size,
            generations=generations,
        )
        configured_legacy_algorithm = lambda items, width: legacy_algorithm(
            items,
            width,
            population_size=population_size,
            generations=generations,
        )
        new_started = time.perf_counter()
        new_result = benchmark(
            case["items"],
            case["machine_width"],
            algorithms={algorithm_name: configured_algorithm},
        )[0]
        new_elapsed = time.perf_counter() - new_started
        print(
            f"    {algorithm_name}: height={new_result['height_mm']:.2f} mm, "
            f"placed={new_result['placed']}, missing={new_result['missing']}, "
            f"elapsed={new_elapsed:.3f}s"
        )
        new_results.append(new_result)

        legacy_started = time.perf_counter()
        legacy_result = benchmark(
            case["items"],
            case["machine_width"],
            algorithms={"Legacy Genetic Algorithm": configured_legacy_algorithm},
        )[0]
        legacy_elapsed = time.perf_counter() - legacy_started
        legacy_total_elapsed += legacy_elapsed
        print(
            f"    Legacy Genetic Algorithm: height={legacy_result['height_mm']:.2f} mm, "
            f"placed={legacy_result['placed']}, missing={legacy_result['missing']}, "
            f"elapsed={legacy_elapsed:.3f}s"
        )
    print(f"Total {algorithm_name} time: {time.perf_counter() - new_total_started:.3f}s")
    print(f"Total Legacy Genetic Algorithm time: {legacy_total_elapsed:.3f}s")
    return new_results


In [ ]:
category_summary_df = (
    results_df
    .groupby(
        ["genetic_height_category", "algorithm"],
        observed=True,
        as_index=False,
    )
    .agg(
        cases=("case_id", "nunique"),
        mean_time_seconds=("seconds", "mean"),
        mean_height_mm=("height_mm", "mean"),
        mean_utilization_percent=("utilization_percent", "mean"),
        mean_placed=("placed", "mean"),
        total_missing=("missing", "sum"),
    )
    .sort_values(["genetic_height_category", "mean_height_mm"])
)

display(
    category_summary_df.style.format({
        "mean_time_seconds": "{:.3f}",
        "mean_height_mm": "{:.2f}",
        "mean_utilization_percent": "{:.2f}",
        "mean_placed": "{:.2f}",
    })
)


In [ ]:
from new_optimizer.benchmark import compare_genetic_packers_batch, run_genetic_algorithm


run_algorithm_sequential(run_genetic_algorithm, 'New Genetic Algorithm', population_size=50, generations=100, mutation_rate=0.15, seed=42, verbose=True, packer="skyline", stagnation_generations=15)


In [1]:
import sys, pandas as pd
from pathlib import Path

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "parameter_search").is_dir()
)
sys.path.insert(0, str(ROOT))

from parameter_search.search import (
    load_and_select_training_cases,
    search_parameters_all,
    search_parameters_by_category,
)
ARTIFACTS = ROOT / "parameter_search/artifacts"
SEARCH = dict(
    n_trials=50,
    packer="skyline",
    time_weight=0.10,
    time_reference_seconds=3.0,
    max_population_size=130,
    max_generations=180,
)

def show_results(rows, title):
    """Show top trials by height quality and by quality + time; return the best of each."""
    df = pd.DataFrame(rows)
    if df.empty:
        return None, None
    by_quality = df.sort_values("quality_objective").reset_index(drop=True)
    by_value = df.sort_values("objective").reset_index(drop=True)
    print(f"\n{title} - jakość bez czasu:")
    display(by_quality.head(10))
    print(f"{title} - pełna ocena: jakość + czas:")
    display(by_value.head(10))
    best_quality, best_value = by_quality.iloc[0].to_dict(), by_value.iloc[0].to_dict()
    print("Best quality:", best_quality)
    print("Best value:", best_value)
    return best_quality, best_value


c:\Users\PC\my-projects\corel-layout-optimizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
training_cases = load_and_select_training_cases(ROOT, per_category=50)
print(f"Training cases selected: {len(training_cases)}")
print(pd.Series([case["size_category"] for case in training_cases]).value_counts().sort_index())

rows, _ = search_parameters_all(
    training_cases,
    results_path=ARTIFACTS / "parameter_search_all_3s_results.json",
    study_path=ARTIFACTS / "parameter_search_all_3s.db",
    **SEARCH,
)
best_quality_parameters, best_value_parameters = show_results(rows, "ALL")

category_results = search_parameters_by_category(
    training_cases,
    results_path=ARTIFACTS / "parameter_search_category_3s_results.json",
    study_path=ARTIFACTS / "parameter_search_category_3s.db",
    **SEARCH,
)
for category, (rows, _) in category_results.items():
    show_results(rows, category.upper())

Training cases selected: 150
medium    50
short     50
tall      50
Name: count, dtype: int64


[I 2026-11-12 10:41:59,741] A new study created in RDB with name: layout_optimizer
[I 2026-11-12 10:46:31,464] Trial 0 finished with value: 0.8900993971448832 and parameters: {'population_size': 80, 'generations': 140, 'mutation_rate': 0.2609671816250754, 'stagnation_generations': 25, 'elite_count': 3, 'crossover_probability': 0.8229470565333281, 'orientation_mutation_rate': 0.12064505859041928, 'swap_mutation_rate': 0.5404751504301438, 'min_width_ratio': 0.9661918489903911, 'mix_categories': 1, 'epsilon': 0.16251050685487964, 'prefer_higher': True, 'acceptance': 0.9851193276585322}. Best is trial 0 with value: 0.8900993971448832.
[I 2026-11-12 10:46:54,385] Trial 1 finished with value: 0.9065464611194345 and parameters: {'population_size': 20, 'generations': 30, 'mutation_rate': 0.057076439104114, 'stagnation_generations': 35, 'elite_count': 5, 'crossover_probability': 0.9350060741234096, 'orientation_mutation_rate': 0.24508221871353572, 'swap_mutation_rate': 0.4895372103191979, 'min_


ALL - jakość bez czasu:


,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,mean_seconds,time_penalty,objective,state,short_quality_objective,short_mean_seconds,medium_quality_objective,medium_mean_seconds,tall_quality_objective,tall_mean_seconds
0,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,1.126412,0.018774,0.886970,COMPLETE,0.986323,0.319196,0.971937,0.958599,0.992051,2.101441
1,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,1.772460,0.029541,0.890099,COMPLETE,0.989908,0.429843,0.971173,1.599568,0.996070,3.287970
2,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.538583,0.008976,0.888967,COMPLETE,0.981714,0.130797,0.973171,0.525307,1.005347,0.959647
3,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,1.352632,0.022544,0.897663,COMPLETE,1.005659,0.242575,0.982098,1.235632,0.996939,2.579688
4,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.133471,0.002225,0.906546,COMPLETE,1.005914,0.039657,0.998167,0.131323,1.017000,0.229432


ALL - pełna ocena: jakość + czas:


,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,mean_seconds,time_penalty,objective,state,short_quality_objective,short_mean_seconds,medium_quality_objective,medium_mean_seconds,tall_quality_objective,tall_mean_seconds
0,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,1.126412,0.018774,0.886970,COMPLETE,0.986323,0.319196,0.971937,0.958599,0.992051,2.101441
1,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.538583,0.008976,0.888967,COMPLETE,0.981714,0.130797,0.973171,0.525307,1.005347,0.959647
2,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,1.772460,0.029541,0.890099,COMPLETE,0.989908,0.429843,0.971173,1.599568,0.996070,3.287970
3,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,1.352632,0.022544,0.897663,COMPLETE,1.005659,0.242575,0.982098,1.235632,0.996939,2.579688
4,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.133471,0.002225,0.906546,COMPLETE,1.005914,0.039657,0.998167,0.131323,1.017000,0.229432


... (11 outputs left out)

,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,epsilon,prefer_higher,acceptance,quality_objective,mean_seconds,time_penalty,objective,state,tall_quality_objective,tall_mean_seconds
0,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,0.197907,True,0.832262,0.992051,2.005871,0.033431,0.896189,COMPLETE,0.992051,2.005871
1,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,0.162511,True,0.985119,0.996070,3.046326,0.050238,0.901487,COMPLETE,0.996070,3.046326
2,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,0.189875,False,0.887406,0.996939,2.632583,0.043876,0.901633,COMPLETE,0.996939,2.632583
3,5,130,90,0.391866,25,4,0.519594,0.085046,0.116108,0.712533,...,0.077237,False,0.938494,1.000468,2.619128,0.043652,0.904787,COMPLETE,1.000468,2.619128
4,39,100,90,0.285661,30,3,0.571295,0.054654,0.199162,0.676229,...,0.052547,False,0.895872,1.002685,1.539944,0.025666,0.904983,COMPLETE,1.002685,1.539944
5,35,90,90,0.137796,35,2,0.722473,0.038872,0.226776,0.763084,...,0.179439,False,0.950643,1.003432,2.679673,0.044661,0.907555,COMPLETE,1.003432,2.679673
6,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.167779,True,0.819220,1.005347,0.954321,0.015905,0.906403,COMPLETE,1.005347,0.954321
7,47,80,90,0.084008,35,3,0.715985,0.025071,0.361443,0.847665,...,0.154014,False,0.849202,1.005651,1.929126,0.032152,0.908301,COMPLETE,1.005651,1.929126
8,9,20,70,0.281061,15,4,0.714384,0.051159,0.214055,0.816587,...,0.123379,False,0.886284,1.013816,0.310978,0.005183,0.912953,COMPLETE,1.013816,0.310978
9,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.041289,False,0.988934,1.017000,0.231352,0.003856,0.915685,COMPLETE,1.017000,0.231352


TALL - pełna ocena: jakość + czas:


,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,epsilon,prefer_higher,acceptance,quality_objective,mean_seconds,time_penalty,objective,state,tall_quality_objective,tall_mean_seconds
0,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,0.197907,True,0.832262,0.992051,2.005871,0.033431,0.896189,COMPLETE,0.992051,2.005871
1,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,0.162511,True,0.985119,0.996070,3.046326,0.050238,0.901487,COMPLETE,0.996070,3.046326
2,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,0.189875,False,0.887406,0.996939,2.632583,0.043876,0.901633,COMPLETE,0.996939,2.632583
3,5,130,90,0.391866,25,4,0.519594,0.085046,0.116108,0.712533,...,0.077237,False,0.938494,1.000468,2.619128,0.043652,0.904787,COMPLETE,1.000468,2.619128
4,39,100,90,0.285661,30,3,0.571295,0.054654,0.199162,0.676229,...,0.052547,False,0.895872,1.002685,1.539944,0.025666,0.904983,COMPLETE,1.002685,1.539944
5,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.167779,True,0.819220,1.005347,0.954321,0.015905,0.906403,COMPLETE,1.005347,0.954321
6,35,90,90,0.137796,35,2,0.722473,0.038872,0.226776,0.763084,...,0.179439,False,0.950643,1.003432,2.679673,0.044661,0.907555,COMPLETE,1.003432,2.679673
7,47,80,90,0.084008,35,3,0.715985,0.025071,0.361443,0.847665,...,0.154014,False,0.849202,1.005651,1.929126,0.032152,0.908301,COMPLETE,1.005651,1.929126
8,9,20,70,0.281061,15,4,0.714384,0.051159,0.214055,0.816587,...,0.123379,False,0.886284,1.013816,0.310978,0.005183,0.912953,COMPLETE,1.013816,0.310978
9,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.041289,False,0.988934,1.017000,0.231352,0.003856,0.915685,COMPLETE,1.017000,0.231352


Best quality: {'trial': 3, 'population_size': 100, 'generations': 30, 'mutation_rate': 0.28336835040598374, 'stagnation_generations': 30, 'elite_count': 2, 'crossover_probability': 0.5644631488274267, 'orientation_mutation_rate': 0.0925485207125623, 'swap_mutation_rate': 0.2500409240184424, 'min_width_ratio': 0.8166747727587942, 'mix_categories': 1, 'epsilon': 0.19790729085066072, 'prefer_higher': True, 'acceptance': 0.8322619035769993, 'quality_objective': 0.9920507130357558, 'mean_seconds': 2.0058708280000612, 'time_penalty': 0.03343118046666769, 'objective': 0.8961887597788469, 'state': 'COMPLETE', 'tall_quality_objective': 0.9920507130357558, 'tall_mean_seconds': 2.0058708280000612}
Best value: {'trial': 3, 'population_size': 100, 'generations': 30, 'mutation_rate': 0.28336835040598374, 'stagnation_generations': 30, 'elite_count': 2, 'crossover_probability': 0.5644631488274267, 'orientation_mutation_rate': 0.0925485207125623, 'swap_mutation_rate': 0.2500409240184424, 'min_width_rat